This script makes docker image, pushes to ECR, and makes a Sagemaker Endpoint instance(s)

In [1]:
#import custom sagemaker deploy tools
import sys
sys.path.append('/home/ec2-user/SageMaker/[REDACTED]AppliedScience/Omar/ModelImplementationWSDK/tools')
from sagemaker_deploy_tools import *

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:172: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[05/27/25 21:38:55] INFO     Found credentials from IAM Role:                                   ]8;id=503608;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=688137;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


[05/27/25 21:38:56] INFO     Found credentials from IAM Role:                                   ]8;id=427304;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=38051;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [35]:
#SET VARIABLES FOR PROJECT HERE

import time
import os
import json

#SET VARIABLES HERE for ECR, model, endpoint, project name
ecr_repository = "[REDACTED]-ml/math-proficiency-model"
endpoint_name = 'math-proficiency-endpoint'
instance_type = 'ml.c5.large'

#aws account vairables
account_id = '[REDACTED]'
region = 'us-east-1'


#scaling parameters for server
min_capacity = 1
max_capacity = 100
target_value = 8  # 55/ Target invocations per instance per minute
scale_in_cooldown = 60  # 5 minutes
scale_out_cooldown = 300  # 5 minutes


#subdirectory model + artifacts live in//current folder name
import os
project_name = os.path.basename(os.getcwd())

# Set variables for Sagemaker Endpoints
image_tag = str(round(time.time()*1000))

#copy variables for docker scripts
AWS_ACCOUNT_ID = account_id
REGION = region
IMAGE_TAG = image_tag
ECR_REPOSITORY = ecr_repository


# Construct the ECR image URI
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{ecr_repository}:{image_tag}'

print('project name: ', project_name, '\nECR Image URI: ', image_uri)

project name:  container 
ECR Image URI:  [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/math-proficiency-model:1748455879806


In [3]:
#test message (for inference testing)
test_message = {
  "skillId": "afa3bfaa479de311b77c005056801da1",
  "questionId": "ff9277e80b8b",
  "eventTime": "2024-11-22 16:20:01.715000+00:00",
  "questionIdsHistory": [
    "ff9277e80b8b",
    "7fa56d98a226",
    "e233aaa276c5"
  ],
  "correctnessHistory": [
    100,
    100,
    100
  ],
  "durationSecondsHistory": [
    6.0,
    5.0,
    4.0
  ],
  "eventTimesHistory": [
    "2024-11-22T16:20:01.000Z",
    "2024-11-22T16:20:01.000Z",
    "2024-11-22T16:20:01.000Z"
  ]
}


In [4]:
#MODIFY YOUR DOCKERFILE TO INCLUDE ALL ARTIFACTS NEEDED
#Check your library depedencies



In [5]:
#buid docker, push to ECR

import subprocess

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    output, error = process.communicate()
    if process.returncode != 0:
        print(f"Error executing command: {command}")
        print(f"Error message: {error.decode('utf-8')}")
        raise Exception(f"Command failed with return code {process.returncode}")
    return output.decode('utf-8')

#look at dockerfile
dockerfile_path = os.path.join('container', 'Dockerfile')
with open(dockerfile_path, 'r') as file:
    dockerfile_contents = file.read()
print("Contents of container/Dockerfile:")
print(dockerfile_contents)

# Change directory to 'container'
os.chdir('container')

# Authenticate Docker to AWS ECR
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin 763104351884.dkr.ecr.{REGION}.amazonaws.com")

# Also authenticate to your own ECR repository
run_command(f"aws ecr get-login-password --region {REGION} | docker login --username AWS --password-stdin {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com")

# Create ECR repository if it doesn't exist
try:
    run_command(f"aws ecr describe-repositories --repository-names {ECR_REPOSITORY}")
except Exception:
    run_command(f"aws ecr create-repository --repository-name {ECR_REPOSITORY}")

# Build the Docker image
run_command(f"docker build -t {ECR_REPOSITORY}:{IMAGE_TAG} .")

# Tag the image for ECR
run_command(f"docker tag {ECR_REPOSITORY}:{IMAGE_TAG} {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")


Contents of container/Dockerfile:
# Start with the AWS Deep Learning Container for Python 3.8
#FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.8.1-cpu-py38-ubuntu20.04
# FROM pytorch/pytorch:1.8.1-cuda11.1-cudnn8-runtime
FROM 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-inference:1.12.1-cpu-py38-ubuntu20.04-sagemaker

# Set working directory
WORKDIR /opt/ml/model

# Install the required packages
RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir \
    numpy \
    scipy \
    pandas \
    flask \
    gunicorn \
    pickle5 
RUN pip install --no-cache-dir -v xgboost==2.1.1
RUN pip install --no-cache-dir joblib==1.4.2
RUN pip install --no-cache-dir scikit-learn
#RUN pip install --no-cache-dir numpy==1.26.4
#RUN pip install --no-cache-dir pandas==1.5.3

# Copy your model file and any other necessary files
COPY proficiency_model.json /opt/ml/model/
#COPY proficiency_scaler.json /opt/ml/model/
COPY confidence_score_scaler.json /opt/

''

In [6]:
#Runs local docker image test before deploy

import requests
import json

def test_docker_inference(repository, tag, test_message):
    try:
        # Run the container
        run_command(f"docker run -d -p 8080:8080 --name test-container {repository}:{tag}")
        print("Container started, waiting for it to be ready...")
        
        import time
        time.sleep(5)
        
        # Test inference request
        inference_response = requests.post(
            'http://localhost:8080/invocations',
            json=test_message,
            headers={'Content-Type': 'application/json'}
        )
        
        print(f"Inference response status: {inference_response.status_code}")
        if inference_response.status_code == 200:
            print("Response:", inference_response.json())
            return True  # Successfully tested
        else:
            raise Exception(f"Test failed with status {inference_response.status_code}: {inference_response.text}")
        
    except Exception as e:
        print(f"Error testing container: {str(e)}")
        raise  # Re-raise the exception to stop notebook execution
    finally:
        run_command("docker stop test-container")
        run_command("docker rm test-container")
        print('cleaned up/removed docker container')
        
# Add after build but before deploy:
test_docker_inference(ECR_REPOSITORY, IMAGE_TAG, test_message)

Container started, waiting for it to be ready...
Inference response status: 200
Response: {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': 5.0, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': 4.0, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DURATIONSECONDS_LAG_1': 1.6094379124341003, 'LOG_DURATIONSECONDS_LAG_10': None, 'LOG_DURATIONSECONDS_LAG_2': 1.3862943611198906, 'LOG_DURATIONSECONDS_LAG_3': None, 'LOG_DURATIONSECONDS_LAG_4': None, 'LOG_DURATIONSECONDS_LAG_5': None, 'LOG_DURATIONSECONDS_LAG_6': None, 'LOG_DURATION

cleaned up/removed docker container


True

In [7]:

# Push the image to ECR
run_command(f"docker push {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

print(f"Image pushed successfully to ECR: {AWS_ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPOSITORY}:{IMAGE_TAG}")

Image pushed successfully to ECR: [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/math-proficiency-model:1748381936386


In [8]:
#Deploy if it doesn't exist; otherwise rolling update; rollback if fails

import boto3
import sagemaker
from sagemaker import get_execution_role
from botocore.exceptions import ClientError

# Initialize the SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Set up the SageMaker session and role
sagemaker_session = sagemaker.Session()
role = get_execution_role()


# Create the model
model = sagemaker.Model(
    image_uri=image_uri,
    role=role
)




# Check if the endpoint exists
if not endpoint_exists(endpoint_name):
    # Deploy the model to create an endpoint
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )

    #Create endpoint
    print(f"Endpoint '{endpoint_name}' is being created. This may take a few minutes...")

    # Wait for the endpoint to be in service
    waiter = boto3.client('sagemaker').get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name)

    print(f"Endpoint '{endpoint_name}' is now in service and ready for real-time inference.")
    
    print(f"Attempting to register scaling.")    
    setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
                       scale_in_cooldown, scale_out_cooldown)
    
    print(f"Finished.")
else:
    #Endpoint exists, do a rolling update/deploy
    print(f"Endpoint '{endpoint_name}' already exists.")
    print('Attempting update...')
    #this function will descale, update, then rescale
    update_endpoint_with_scaling(endpoint_name, image_uri, test_message, min_capacity, max_capacity, target_value, scale_in_cooldown, scale_out_cooldown)
    

Endpoint 'math-proficiency-endpoint' already exists.
Attempting update...
Current endpoint status: InService
Attempting deregistration of scalable target...
Deregistered scalable target for endpoint: math-proficiency-endpoint
Attempting updating of endpoint...
Current Endpoint Config: math-20250429012739
Current Model Name: math-proficiency-model-2025-04-29-01-27-39-085
Current Image URI: [REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/math-proficiency-model:1745890041982
New config name:  math-20250527214413  Old config name:  math-20250429012739
Endpoint 'math-proficiency-endpoint' is being updated. This may take a few minutes...
Endpoint 'math-proficiency-endpoint' has been successfully updated with the new model.
Attempting registration of scaling of target...
No existing scaling policy found
No existing scalable target found
Auto-scaling has been set up for endpoint: math-proficiency-endpoint
Min capacity: 1
Max capacity: 100
Target value: 15 invocations per instance per 

Unit Tests (bunch of test messages)

In [9]:
#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': 'afa3bfaa479de311b77c005056801da1', 'questionId': 'ff9277e80b8b', 'eventTime': '2024-11-22 16:20:01.715000+00:00', 'questionIdsHistory': ['ff9277e80b8b', '7fa56d98a226', 'e233aaa276c5'], 'correctnessHistory': [100, 100, 100], 'durationSecondsHistory': [6.0, 5.0, 4.0], 'eventTimesHistory': ['2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z']}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 1.0, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': 5.0, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': 4.0, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None

In [10]:
#test endpoint with double json-ifciation

try:

    runtime = boto3.client('sagemaker-runtime')
    print('submitting: ', test_message)
    response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                       ContentType='application/json', 
                                       Body=json.dumps(str(json.dumps(test_message))))
    result = json.loads(response['Body'].read().decode())

    print('unsuccessfully failed.')
    print('result: ', result)
    
except Exception as e:
    print('successfully failed.')
    print('error: ', e)

submitting:  {'skillId': 'afa3bfaa479de311b77c005056801da1', 'questionId': 'ff9277e80b8b', 'eventTime': '2024-11-22 16:20:01.715000+00:00', 'questionIdsHistory': ['ff9277e80b8b', '7fa56d98a226', 'e233aaa276c5'], 'correctnessHistory': [100, 100, 100], 'durationSecondsHistory': [6.0, 5.0, 4.0], 'eventTimesHistory': ['2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z']}
successfully failed.
error:  An error occurred (ModelError) when calling the InvokeEndpoint operation: Received server error (500) from primary and could not load the entire response body. See https://us-east-1.console.aws.amazon.com/cloudwatch/home?region=us-east-1#logEventViewer:group=/aws/sagemaker/Endpoints/math-proficiency-endpoint in account [REDACTED] for more information.


In [11]:
#test endpoint with non json-ifciation
import boto3
import json

try:
    runtime = boto3.client('sagemaker-runtime')
    print('submitting: ', test_message)
    response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                       ContentType='application/json', 
                                       Body=test_message)
    result = json.loads(response['Body'].read().decode())

    print('result: ', result)
except Exception as e:
    print('successfully failed.')
    print('error: ', e)

submitting:  {'skillId': 'afa3bfaa479de311b77c005056801da1', 'questionId': 'ff9277e80b8b', 'eventTime': '2024-11-22 16:20:01.715000+00:00', 'questionIdsHistory': ['ff9277e80b8b', '7fa56d98a226', 'e233aaa276c5'], 'correctnessHistory': [100, 100, 100], 'durationSecondsHistory': [6.0, 5.0, 4.0], 'eventTimesHistory': ['2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z']}
successfully failed.
error:  Parameter validation failed:
Invalid type for parameter Body, value: {'skillId': 'afa3bfaa479de311b77c005056801da1', 'questionId': 'ff9277e80b8b', 'eventTime': '2024-11-22 16:20:01.715000+00:00', 'questionIdsHistory': ['ff9277e80b8b', '7fa56d98a226', 'e233aaa276c5'], 'correctnessHistory': [100, 100, 100], 'durationSecondsHistory': [6.0, 5.0, 4.0], 'eventTimesHistory': ['2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z', '2024-11-22T16:20:01.000Z']}, type: <class 'dict'>, valid types: <class 'bytes'>, <class 'bytearray'>, file-like object


In [12]:
test_message_multiple_events = [{'skillId': 'FAKE_MBWLT',
  'questionId': 'question_50',
  'eventTime': '2025-03-23T10:21:03.150912',
  'questionIdsHistory': ['question_254',
   'question_578',
   'question_824',
   'question_79',
   'question_628',
   'question_795',
   'question_243',
   'question_997',
   'question_583',
   'question_611'],
  'correctnessHistory': [100, 100, 100, 100, 100, 0, 0, 100, 0, 0],
  'durationSecondsHistory': [136, 86, 181, 256, 44, 218, 214, 256, 280, 139],
  'eventTimesHistory': ['2025-03-23T10:21:03.150912',
   '2025-03-23T10:11:03.150912',
   '2025-03-23T10:01:03.150912',
   '2025-03-23T09:51:03.150912',
   '2025-03-23T09:41:03.150912',
   '2025-03-23T09:31:03.150912',
   '2025-03-23T09:21:03.150912',
   '2025-03-23T09:11:03.150912',
   '2025-03-23T09:01:03.150912',
   '2025-03-23T08:51:03.150912']},
 {'skillId': 'FAKE_4J20C',
  'questionId': 'question_682',
  'eventTime': '2025-03-23T10:21:03.150964',
  'questionIdsHistory': ['question_929',
   'question_669',
   'question_666',
   'question_227',
   'question_486',
   'question_380',
   'question_385',
   'question_375',
   'question_775',
   'question_255'],
  'correctnessHistory': [0, 0, 0, 100, 0, 100, 0, 100, 100, 100],
  'durationSecondsHistory': [97, 270, 47, 78, 111, 45, 62, 134, 258, 278],
  'eventTimesHistory': ['2025-03-23T10:21:03.150964',
   '2025-03-23T10:11:03.150964',
   '2025-03-23T10:01:03.150964',
   '2025-03-23T09:51:03.150964',
   '2025-03-23T09:41:03.150964',
   '2025-03-23T09:31:03.150964',
   '2025-03-23T09:21:03.150964',
   '2025-03-23T09:11:03.150964',
   '2025-03-23T09:01:03.150964',
   '2025-03-23T08:51:03.150964']},
 {'skillId': '8e051784e51f413d83565054be4123d6',
  'questionId': 'question_331',
  'eventTime': '2025-03-23T10:21:03.151009',
  'questionIdsHistory': ['question_462',
   'question_128',
   'question_675',
   'question_824',
   'question_980',
   'question_437',
   'question_882',
   'question_979',
   'question_894',
   'question_707'],
  'correctnessHistory': [0, 100, 100, 100, 100, 0, 0, 0, 0, 0],
  'durationSecondsHistory': [128, 211, 192, 102, 161, 208, 11, 236, 198, 179],
  'eventTimesHistory': ['2025-03-23T10:21:03.151009',
   '2025-03-23T10:11:03.151009',
   '2025-03-23T10:01:03.151009',
   '2025-03-23T09:51:03.151009',
   '2025-03-23T09:41:03.151009',
   '2025-03-23T09:31:03.151009',
   '2025-03-23T09:21:03.151009',
   '2025-03-23T09:11:03.151009',
   '2025-03-23T09:01:03.151009',
   '2025-03-23T08:51:03.151009']},
 {'skillId': 'FAKE_MBWLT',
  'questionId': 'question_253',
  'eventTime': '2025-03-23T10:21:03.151052',
  'questionIdsHistory': ['question_253',
   'question_476',
   'question_3',
   'question_725',
   'question_279',
   'question_270',
   'question_7',
   'question_353',
   'question_150',
   'question_333'],
  'correctnessHistory': [100, 100, 100, 100, 100, 100, 100, 0, 100, 100],
  'durationSecondsHistory': [115, 137, 142, 105, 166, 126, 31, 58, 173, 144],
  'eventTimesHistory': ['2025-03-23T10:21:03.151052',
   '2025-03-23T10:11:03.151052',
   '2025-03-23T10:01:03.151052',
   '2025-03-23T09:51:03.151052',
   '2025-03-23T09:41:03.151052',
   '2025-03-23T09:31:03.151052',
   '2025-03-23T09:21:03.151052',
   '2025-03-23T09:11:03.151052',
   '2025-03-23T09:01:03.151052',
   '2025-03-23T08:51:03.151052']},
 {'skillId': 'FAKE_JGCAG',
  'questionId': 'question_866',
  'eventTime': '2025-03-23T10:21:03.151109',
  'questionIdsHistory': ['question_338',
   'question_176',
   'question_204',
   'question_801',
   'question_38',
   'question_837',
   'question_486',
   'question_691',
   'question_223',
   'question_574'],
  'correctnessHistory': [0, 0, 0, 0, 0, 0, 100, 0, 0, 100],
  'durationSecondsHistory': [57, 256, 10, 89, 191, 161, 227, 200, 156, 134],
  'eventTimesHistory': ['2025-03-23T10:21:03.151109',
   '2025-03-23T10:11:03.151109',
   '2025-03-23T10:01:03.151109',
   '2025-03-23T09:51:03.151109',
   '2025-03-23T09:41:03.151109',
   '2025-03-23T09:31:03.151109',
   '2025-03-23T09:21:03.151109',
   '2025-03-23T09:11:03.151109',
   '2025-03-23T09:01:03.151109',
   '2025-03-23T08:51:03.151109']}]


#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

print('result: ', result)
print('PREDICTION: ', result['prediction'])

submitting:  [{'skillId': 'FAKE_MBWLT', 'questionId': 'question_50', 'eventTime': '2025-03-23T10:21:03.150912', 'questionIdsHistory': ['question_254', 'question_578', 'question_824', 'question_79', 'question_628', 'question_795', 'question_243', 'question_997', 'question_583', 'question_611'], 'correctnessHistory': [100, 100, 100, 100, 100, 0, 0, 100, 0, 0], 'durationSecondsHistory': [136, 86, 181, 256, 44, 218, 214, 256, 280, 139], 'eventTimesHistory': ['2025-03-23T10:21:03.150912', '2025-03-23T10:11:03.150912', '2025-03-23T10:01:03.150912', '2025-03-23T09:51:03.150912', '2025-03-23T09:41:03.150912', '2025-03-23T09:31:03.150912', '2025-03-23T09:21:03.150912', '2025-03-23T09:11:03.150912', '2025-03-23T09:01:03.150912', '2025-03-23T08:51:03.150912']}, {'skillId': 'FAKE_4J20C', 'questionId': 'question_682', 'eventTime': '2025-03-23T10:21:03.150964', 'questionIdsHistory': ['question_929', 'question_669', 'question_666', 'question_227', 'question_486', 'question_380', 'question_385', 'ques

In [13]:
#test endpoint w multipile messages
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  [{'skillId': 'FAKE_MBWLT', 'questionId': 'question_50', 'eventTime': '2025-03-23T10:21:03.150912', 'questionIdsHistory': ['question_254', 'question_578', 'question_824', 'question_79', 'question_628', 'question_795', 'question_243', 'question_997', 'question_583', 'question_611'], 'correctnessHistory': [100, 100, 100, 100, 100, 0, 0, 100, 0, 0], 'durationSecondsHistory': [136, 86, 181, 256, 44, 218, 214, 256, 280, 139], 'eventTimesHistory': ['2025-03-23T10:21:03.150912', '2025-03-23T10:11:03.150912', '2025-03-23T10:01:03.150912', '2025-03-23T09:51:03.150912', '2025-03-23T09:41:03.150912', '2025-03-23T09:31:03.150912', '2025-03-23T09:21:03.150912', '2025-03-23T09:11:03.150912', '2025-03-23T09:01:03.150912', '2025-03-23T08:51:03.150912']}, {'skillId': 'FAKE_4J20C', 'questionId': 'question_682', 'eventTime': '2025-03-23T10:21:03.150964', 'questionIdsHistory': ['question_929', 'question_669', 'question_666', 'question_227', 'question_486', 'question_380', 'question_385', 'ques

In [14]:
result['prediction']

{'item_prediction': [0.41478389501571655,
  0.15989187359809875,
  0.3210444748401642,
  0.34129318594932556,
  0.17637860774993896],
 'item_prediction_confidence': [0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502,
  9.867217040076751],
 'item_prediction_error': [0.5922628045082092,
  0.5493558645248413,
  0.5812946557998657,
  0.5482966899871826,
  0.39830082654953003],
 'model_version': ['5.05', '5.05', '5.05', '5.05', '5.05'],
 'skill_prediction': [0.43608665466308594,
  0.19320142269134521,
  0.30741962790489197,
  0.612372875213623,
  0.30632129311561584],
 'skill_prediction_confidence': [0.0008062110499286502,
  0.0008062110499286502,
  0.0008062110499286502,
  9.768859291985455,
  0.0008062110499286502],
 'skill_prediction_error': [0.5903750658035278,
  0.5179034471511841,
  0.5902139544487,
  0.45308300852775574,
  0.5634634494781494]}

In [15]:
#test endpoint with empty time message
empty_test_message = {'skillId': '8e051784e51f413d83565054be4123d6', 
                    'questionId': 'question_478', 
                    'eventTime': '',
                    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                                           'question_367', 
                                           'question_465', 'question_184', 'question_283', 'question_327'], 
                    'correctnessHistory': [], 
                    'durationSecondsHistory': [], 
                    'eventTimesHistory': []}


import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(empty_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': 'question_478', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': None, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DU

In [16]:
#test endpoint with empty qid message & empty event time
empty_test_message = {'skillId': '8e051784e51f413d83565054be4123d6', 
                    'questionId': '', 
                    'eventTime': '',
                    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                                           'question_367', 
                                           'question_465', 'question_184', 'question_283', 'question_327'], 
                    'correctnessHistory': [], 
                    'durationSecondsHistory': [], 
                    'eventTimesHistory': []}


import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(empty_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': '', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': None, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DURATIONSECOND

In [17]:
reproduced_test_message = {
    'skillId': '8e051784e51f413d83565054be4123d6',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-12-10 13:34:29.371403',
    'questionIdsHistory': [], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': '', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': None, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DURATIONSECOND

In [18]:
reproduced_test_message = {
    'skillId': 'sqyudjgp',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-10-04T19:15:18.187471',
    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                           'question_367', 
                           'question_465', 'question_184', 'question_283', 'question_327'], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': '', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': None, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DURATIONSECOND

In [19]:
 reproduced_test_message = {
    'skillId': 'sqyudjgp',
    'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca',
    'eventTime': '2024-10-04T19:15:18.187471',
    'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 
                           'question_465', 'question_184', 'question_283', 'question_327'], 
    'correctnessHistory': [],
    'durationSecondsHistory': [],
    'eventTimesHistory': []
}

print('submitting: ', empty_test_message)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_test_message))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': '', 'eventTime': '', 'questionIdsHistory': ['question_442', 'question_801', 'question_633', 'question_616', 'question_367', 'question_465', 'question_184', 'question_283', 'question_327'], 'correctnessHistory': [], 'durationSecondsHistory': [], 'eventTimesHistory': []}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': None, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': None, 'CORRECTNESS_LAG_3': None, 'CORRECTNESS_LAG_4': None, 'CORRECTNESS_LAG_5': None, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'DURATIONSECONDS_LAG_6': None, 'DURATIONSECONDS_LAG_7': None, 'DURATIONSECONDS_LAG_8': None, 'DURATIONSECONDS_LAG_9': None, 'LOG_DURATIONSECOND

In [20]:
reproduced_message_2 = {"correctnessHistory": [0, 1, 0, 1, 0.75, 1], 
                        "durationSecondsHistory": [None, None, None, None, None, None], 
                        "eventTime": "2024-12-10T13:34:29.371403527Z", 
                        "eventTimesHistory": ["2024-12-10T13:34:29.371403527Z", "2024-12-10T13:34:20.931262655Z", None, None, None, None], "questionId": "6dc12d68-af7f-441f-9839-5882e115a2ca", 
                        "questionIdsHistory": ["6dc12d68-af7f-441f-9839-5882e115a2ca", "5c686c0c-d332-400e-b621-a8de2d0f824f", None, None, None, None], "skillId": "sqyudjgp"}


print('submitting: ', reproduced_message_2)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(reproduced_message_2))
result = json.loads(response['Body'].read().decode())

print('result: ', result)

submitting:  {'correctnessHistory': [0, 1, 0, 1, 0.75, 1], 'durationSecondsHistory': [None, None, None, None, None, None], 'eventTime': '2024-12-10T13:34:29.371403527Z', 'eventTimesHistory': ['2024-12-10T13:34:29.371403527Z', '2024-12-10T13:34:20.931262655Z', None, None, None, None], 'questionId': '6dc12d68-af7f-441f-9839-5882e115a2ca', 'questionIdsHistory': ['6dc12d68-af7f-441f-9839-5882e115a2ca', '5c686c0c-d332-400e-b621-a8de2d0f824f', None, None, None, None], 'skillId': 'sqyudjgp'}
result:  {'debug_info': {'feature_engineered_input': {'CORRECTNESS_LAG_1': 1.0, 'CORRECTNESS_LAG_10': None, 'CORRECTNESS_LAG_2': 0.0, 'CORRECTNESS_LAG_3': 1.0, 'CORRECTNESS_LAG_4': 0.75, 'CORRECTNESS_LAG_5': 1.0, 'CORRECTNESS_LAG_6': None, 'CORRECTNESS_LAG_7': None, 'CORRECTNESS_LAG_8': None, 'CORRECTNESS_LAG_9': None, 'DURATIONSECONDS_LAG_1': None, 'DURATIONSECONDS_LAG_10': None, 'DURATIONSECONDS_LAG_2': None, 'DURATIONSECONDS_LAG_3': None, 'DURATIONSECONDS_LAG_4': None, 'DURATIONSECONDS_LAG_5': None, 'D

In [21]:
result['prediction']

{'item_prediction': [0.4638093411922455],
 'item_prediction_confidence': [6.252166692196683],
 'item_prediction_error': [0.4671953022480011],
 'model_version': ['5.05'],
 'skill_prediction': [0.5784821510314941],
 'skill_prediction_confidence': [6.313438731991261],
 'skill_prediction_error': [0.46144720911979675]}

In [22]:
test_message_multiple_events = [{'skillId': '8e051784e51f413d83565054be4123d6',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 100, 100, 0, 0, 0, 0, 0],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '8e051784e51f413d83565054be4123d6',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [0, 0, 0, 0, 0, 100, 100, 100, 100],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '8e051784e51f413d83565054be4123d6',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [0, 0, 100, 100, 100, 100, 0, 0, 0],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []},
{'skillId': '8e051784e51f413d83565054be4123d6',
  'questionId': 'question_136',
  'eventTime': '2024-10-04T19:15:18.187471',
  'questionIdsHistory': ['question_852',
   'question_204',
   'question_75',
   'question_862',
   'question_775',
   'question_649',
   'question_928',
   'question_242',
   'question_322'],
  'correctnessHistory': [100, 100, 0, 0, 0, 0, 0, 100, 100],
  'durationSecondsHistory': [None, None,None,None,None,None,None,None,None],
  'eventTimesHistory': []}]


#test endpoint
import boto3
import json

runtime = boto3.client('sagemaker-runtime')
print('submitting: ', test_message_multiple_events)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(test_message_multiple_events))
result = json.loads(response['Body'].read().decode())

# print('result: ', result)
result['prediction']

submitting:  [{'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [100, 100, 100, 100, 0, 0, 0, 0, 0], 'durationSecondsHistory': [None, None, None, None, None, None, None, None, None], 'eventTimesHistory': []}, {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': 'question_136', 'eventTime': '2024-10-04T19:15:18.187471', 'questionIdsHistory': ['question_852', 'question_204', 'question_75', 'question_862', 'question_775', 'question_649', 'question_928', 'question_242', 'question_322'], 'correctnessHistory': [0, 0, 0, 0, 0, 100, 100, 100, 100], 'durationSecondsHistory': [None, None, None, None, None, None, None, None, None], 'eventTimesHistory': []}, {'skillId': '8e051784e51f413d83565054be4123d6', 'questionId': 'question_136'

{'item_prediction': [0.5985517501831055,
  0.33139708638191223,
  0.5774446129798889,
  0.40445539355278015],
 'item_prediction_confidence': [9.768859291985455,
  58.16409619710248,
  70.93367301692237,
  0.6344880962938478],
 'item_prediction_error': [0.4483805000782013,
  0.4367293417453766,
  0.42134469747543335,
  0.4573425054550171],
 'model_version': ['5.05', '5.05', '5.05', '5.05'],
 'skill_prediction': [0.6045742034912109,
  0.3080017566680908,
  0.4266963601112366,
  0.6000544428825378],
 'skill_prediction_confidence': [9.864798406926965,
  18.467070309665665,
  1.2778445141369106,
  2.853987116747422],
 'skill_prediction_error': [0.4249984622001648,
  0.38609299063682556,
  0.478181928396225,
  0.4338648021221161]}

In [23]:
inf_test =  test_input = {
  "skillId": "evnldqti",
  "questionId": "94046f4d-7f58-4ed6-ab60-ca6edfc300f4",
  "eventTime": "2024-06-05 07:03:58.636000+00:00",
  "questionIdsHistory": [
    "94046f4d-7f58-4ed6-ab60-ca6edfc300f4",
    "fe63aeb7-9d8e-43a5-857b-6ce7d4a38a63",
    "f8187d97-6abf-4d5d-8556-eb4416126031",
    "e3f28d4d-43f1-475e-9ecf-b961aac28ed7",
    "d330d5a4-b374-47f3-a40a-1180518cbe8f",
    "cdd3cff8-3fa0-4a0f-a72e-f0272702950d",
    "b22aba47-c2a5-4868-8e52-6be05eb7732a",
    "af90d47d-7a56-4653-90b1-3791233f7be3",
    "9dc1166a-f460-41ec-8dbb-44b1f7b8e477",
    "888949ce-ca06-452d-8813-3275b51212d5",
    "7363d704-a28e-417b-b6bc-840864f791ef"
  ],
  "correctnessHistory": [
    0,
    100,
    100,
    100,
    100,
    0,
    0,
    100,
    100,
    100,
    0
  ],
  "durationSecondsHistory": [],
  "eventTimesHistory": [
    "2024-06-05T07:03:58.000Z",
    "2024-05-15T07:10:32.000Z",
    "2024-05-15T07:10:27.000Z",
    "2024-05-15T07:10:23.000Z",
    "2024-05-15T07:10:19.000Z",
    "2024-05-15T07:10:08.000Z",
    "2024-05-15T07:09:49.000Z",
    "2024-05-15T07:09:40.000Z",
    "2024-05-15T07:09:34.000Z",
    "2024-05-15T07:09:31.000Z",
    "2024-05-15T07:07:57.000Z"
  ]
}


import boto3
runtime = boto3.client('sagemaker-runtime')
print('submitting: ', inf_test)
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(inf_test))
result = json.loads(response['Body'].read().decode())

print('result: ', result)
result['prediction']

submitting:  {'skillId': 'evnldqti', 'questionId': '94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'eventTime': '2024-06-05 07:03:58.636000+00:00', 'questionIdsHistory': ['94046f4d-7f58-4ed6-ab60-ca6edfc300f4', 'fe63aeb7-9d8e-43a5-857b-6ce7d4a38a63', 'f8187d97-6abf-4d5d-8556-eb4416126031', 'e3f28d4d-43f1-475e-9ecf-b961aac28ed7', 'd330d5a4-b374-47f3-a40a-1180518cbe8f', 'cdd3cff8-3fa0-4a0f-a72e-f0272702950d', 'b22aba47-c2a5-4868-8e52-6be05eb7732a', 'af90d47d-7a56-4653-90b1-3791233f7be3', '9dc1166a-f460-41ec-8dbb-44b1f7b8e477', '888949ce-ca06-452d-8813-3275b51212d5', '7363d704-a28e-417b-b6bc-840864f791ef'], 'correctnessHistory': [0, 100, 100, 100, 100, 0, 0, 100, 100, 100, 0], 'durationSecondsHistory': [], 'eventTimesHistory': ['2024-06-05T07:03:58.000Z', '2024-05-15T07:10:32.000Z', '2024-05-15T07:10:27.000Z', '2024-05-15T07:10:23.000Z', '2024-05-15T07:10:19.000Z', '2024-05-15T07:10:08.000Z', '2024-05-15T07:09:49.000Z', '2024-05-15T07:09:40.000Z', '2024-05-15T07:09:34.000Z', '2024-05-15T07:09:31.

{'item_prediction': [0.5920702219009399],
 'item_prediction_confidence': [6.313438731991261],
 'item_prediction_error': [0.46042266488075256],
 'model_version': ['5.05'],
 'skill_prediction': [0.4733582139015198],
 'skill_prediction_confidence': [6.2513604811467545],
 'skill_prediction_error': [0.5001819133758545]}

In [24]:
# CHECK FOR NANs because python won't error out but Haskell will

In [25]:
def check_for_nans(response_body):
    return 'nan' in str(response_body)

# Use it after your API call:
response = runtime.invoke_endpoint(EndpointName=endpoint_name, 
                                   ContentType='application/json', 
                                   Body=json.dumps(inf_test))
result = json.loads(response['Body'].read().decode())
assert not check_for_nans(result), "Found NaN values in response - will cause Haskell server error"

Cloudwatch metrics, instance counts, and scaling progress

In [26]:
import boto3
from datetime import datetime, timedelta

cloudwatch = boto3.client('cloudwatch')

response = cloudwatch.get_metric_statistics(
    Namespace='AWS/SageMaker',
    MetricName='Invocations',
    Dimensions=[{'Name': 'EndpointName', 'Value': endpoint_name}],
    StartTime=datetime.utcnow() - timedelta(hours=24),
    EndTime=datetime.utcnow(),
    Period=60,
    Statistics=['Sum']
)
response

{'Label': 'Invocations',
 'Datapoints': [],
 'ResponseMetadata': {'RequestId': 'c3510f4b-5823-4ef8-b4ea-aafbc5644bf1',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'c3510f4b-5823-4ef8-b4ea-aafbc5644bf1',
   'content-type': 'text/xml',
   'content-length': '334',
   'date': 'Tue, 27 May 2025 21:49:22 GMT'},
  'RetryAttempts': 0}}

In [27]:
import boto3

client = boto3.client('sagemaker')

response = client.describe_endpoint(
    EndpointName='[REDACTED]'
)

print('scaling policy:')
print(json.dumps(response, indent=2, default=str))

app_scaling = boto3.client('application-autoscaling')

try:
    response = app_scaling.describe_scaling_policies(
        ServiceNamespace='sagemaker',
        ResourceId=f'endpoint/[REDACTED]/variant/AllTraffic'
    )
    print('\n\ncurrent instance count/desired instance: ')
    print(json.dumps(response, indent=2, default=str))
except Exception as e:
    print("Error:", e)
    
    

scaling policy:
{
  "EndpointName": "[REDACTED]",
  "EndpointArn": "arn:aws:sagemaker:us-east-1:[REDACTED]:endpoint/[REDACTED]",
  "EndpointConfigName": "ela-20250119212457",
  "ProductionVariants": [
    {
      "VariantName": "AllTraffic",
      "DeployedImages": [
        {
          "SpecifiedImage": "[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model:1737321886461",
          "ResolvedImage": "[REDACTED].dkr.ecr.us-east-1.amazonaws.com/[REDACTED]-ml/ela-proficiency-model@sha256:7a7b5403128c23d21b4519880fdc8cc655b2fea79602edced1e595728c3525d9",
          "ResolutionTime": "2025-05-27 21:46:52.268000+00:00"
        }
      ],
      "CurrentWeight": 1.0,
      "DesiredWeight": 1.0,
      "CurrentInstanceCount": 13,
      "DesiredInstanceCount": 13
    }
  ],
  "EndpointStatus": "Updating",
  "CreationTime": "2024-10-04 19:24:47.157000+00:00",
  "LastModifiedTime": "2025-05-27 21:48:51.925000+00:00",
  "ResponseMetadata": {
    "RequestId": "e43e51b8-8072-4

In [38]:
# # ADJUST PARAMTERS AT TOP OF NOTEBOOK TO MAINTAIN CONSISTENCY, 
# # FUNCTION BELOW REQUIRES VARIABLES ABOVE
# update_endpoint_with_scaling(
#     endpoint_name=endpoint_name,
#     new_image_uri=image_uri,
#     test_message=test_message,
#     min_capacity=min_capacity,
#     max_capacity=max_capacity,
#     target_value=target_value)

In [37]:
setup_auto_scaling(endpoint_name, min_capacity, max_capacity, target_value, 
                       scale_in_cooldown, scale_out_cooldown)

Deleted existing scaling policy for endpoint: math-proficiency-endpoint
Deregistered existing scalable target
Auto-scaling has been set up for endpoint: math-proficiency-endpoint
Min capacity: 1
Max capacity: 100
Target value: 8 invocations per instance per minute
Scale-in cooldown: 60 seconds
Scale-out cooldown: 300 seconds


In [39]:
verify_scaling_policy(endpoint_name)

Current endpoint configuration:
Instance Type: ml.c5.large
Min Capacity: 1
Max Capacity: 100

Scaling policy configuration:
Policy Name: ScalingPolicy-math-proficiency-endpoint
Target Value: 8.0
Scale-in Cooldown: 60 seconds
Scale-out Cooldown: 300 seconds


In [42]:
check_scaling_activities(endpoint_name, MaxResults=500)

Recent scaling activities:

Activity ID: b4996052-2757-4b47-ba0c-878f60c56832
Status: Successful
Description: Setting desired instance count to 37.
Cause: monitor alarm TargetTracking-endpoint/math-proficiency-endpoint/variant/AllTraffic-AlarmHigh-b7ddf541-0c82-476d-a81a-abccf8626fdc in state ALARM triggered policy ScalingPolicy-math-proficiency-endpoint
Start Time: 2025-05-28 18:39:16.855000+00:00
Status Message: Successfully set desired instance count to 37. Change successfully fulfilled by sagemaker.

Activity ID: a734af01-7bae-4a7e-b135-67552139e100
Status: Successful
Description: Setting desired instance count to 36.
Cause: monitor alarm TargetTracking-endpoint/math-proficiency-endpoint/variant/AllTraffic-AlarmHigh-b7ddf541-0c82-476d-a81a-abccf8626fdc in state ALARM triggered policy ScalingPolicy-math-proficiency-endpoint
Start Time: 2025-05-28 18:24:16.789000+00:00
Status Message: Successfully set desired instance count to 36. Change successfully fulfilled by sagemaker.

Activity